# Electrode drawings

A two-grid extraction gap as a dimensioned drawing (plasma-plots' `plot_domain`), then its vacuum field from the panel method, with the electrodes drawn over the potential.

The electrodes are simgeometry shapes: each grid is an `Aperture2D` slit, two plates 0.3 mm thick around a gap. `plot_domain` draws them as a technical drawing, with chains of dimensions along both axes taken from the geometry itself. The shapes are built in metres and labelled in millimetres through `scale` and `unit`.

In [ ]:
import numpy as np
import plasma_plots
from plasma_plots import plot_domain
from plasma_plots.geometry import draw_geometry
from simgeometry import Aperture2D

from tree_d_3d import BoundaryIntegralElectrodes, conductors_from_shape

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex")

mm = 1e-3
electrodes = {
    "plasma grid, 0 V": (Aperture2D.slit(1.0 * mm, 0.4 * mm, 0.3 * mm, 1.6 * mm), 0.0),
    "extraction grid, 5 kV": (
        Aperture2D.slit(2.5 * mm, 0.6 * mm, 0.3 * mm, 1.6 * mm),
        5e3,
    ),
}
bounds = ((0.0, -2.2 * mm), (4.0 * mm, 2.2 * mm))
plot_domain(
    {name: shape for name, (shape, _) in electrodes.items()},
    bounds=bounds,
    scale=1e3,
    unit="mm",
    title="The extraction gap",
)

The same shapes become conductors for the panel method: `conductors_from_shape` puts panels on each plate's outline, 25 µm long at the aperture edges and up to 0.1 mm away from them. The vacuum field is then tabulated on a grid, and the electrodes are drawn over it with `draw_geometry`.

In [ ]:
def panel_length(points):
    """25 µm at the plates' edges facing the beam, growing to 0.1 mm."""
    distance = np.min(
        np.abs(np.abs(points[1])[:, None] - np.array([0.4, 0.6]) * mm), axis=1
    )
    return np.minimum(25e-6 + 0.15 * distance, 0.1 * mm)


conductors = [
    conductor
    for shape, potential in electrodes.values()
    for conductor in conductors_from_shape(shape, potential, panel_length)
]
solution = BoundaryIntegralElectrodes(conductors)
solution.solve()
table = solution.tabulate((0.0, -2.2 * mm), (4.0 * mm, 2.2 * mm), (161, 177))
print(f"{solution.num_panels} panels")

with plasma_plots.figure(1, 1, figsize=(6.4, 6.4)) as fig:
    ax = fig[0]
    mesh = ax.contourf(
        table.x / mm, table.y / mm, table.potential.T / 1e3, levels=24, cmap="viridis"
    )
    ax.figure.colorbar(mesh, ax=ax, label=r"$\phi\ (\mathrm{kV})$")
    draw_geometry(
        ax, [shape for shape, _ in electrodes.values()], scale=1e3, color="0.85"
    )
    ax.set_aspect("equal")
    ax.set(
        xlabel=r"$x\ (\mathrm{mm})$",
        ylabel=r"$y\ (\mathrm{mm})$",
        title="Vacuum potential",
    )
fig

Along the beam axis ($y = 0$) the field peaks between the grids. At $y = 0.3$ mm, 0.1 mm inside the plasma grid's aperture, it is stronger and changes faster: next to the plate's edges, where the panels have to be short.

In [ ]:
x = np.linspace(0.0, 4.0 * mm, 400)
with plasma_plots.figure(1, 1, figsize=(6.4, 4.2)) as fig:
    for height, label in ((0.0, "on the axis"), (0.3 * mm, "at y = 0.3 mm")):
        _, field = solution.probe(np.stack((x, np.full_like(x, height))))
        fig[0].plot(x / mm, field[0] / 1e6, label=label)
    fig[0].set(xlabel=r"$x\ (\mathrm{mm})$", ylabel=r"$E_x\ (\mathrm{MV/m})$")
    fig[0].legend()
fig